<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 9 — Numbers and Pictures

Session 8 ended with a clean table. This session is about **describing** one:
in a few numbers, and in a picture. They are two views of the same question,
and each covers the other's blind spots. A summary number hides the shape of
the data. A picture shows the shape and cannot be put in a sentence.

Six ideas, one part each:

| Part | Idea | In one sentence |
|---|---|---|
| 1 | Centre | The mean and the median answer different questions, and under skew they give different answers. |
| 2 | Spread | Variance, standard deviation and the interquartile range say how far the values lie from the centre. |
| 3 | Outliers | A value far from the rest is a question to investigate, not a row to delete. |
| 4 | Your first figure | A Figure holds Axes; you draw by calling methods of the Axes. |
| 5 | The chart follows the question | Distribution, comparison, relationship, trend: four questions, four charts. |
| 6 | Honest pictures | A figure without labels and units is not yet a result, and an axis can mislead. |

**How this session's notebooks fit together**

| Notebook | What it is |
|---|---|
| `01_Numbers_and_Pictures` (this one) | The ideas, on data small enough to read |
| `02_Consumer_Complaints_Example` | One real dataset described from raw file to finished figure |
| `03_Statistics_and_Plotting_Reference` | A lookup guide to keep open while you do Homework 8 |

**What is not repeated here.** Loading, inspecting and cleaning a table,
missing values, and `groupby` are Session 8
(`../08-Python-Data-Analysis/`). Selecting, joining and dates are Session 7
(`../07-Data-Tables/`). This session uses them without explaining them again.

**Before you start**

- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**. Before each cell, guess what
  it will show.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

---

## Part 1 — Centre: one number for many

A small company has nine employees. Their salaries, in thousands of dollars a
year:

In [ ]:
staff = pd.Series([38, 41, 42, 45, 47, 50, 52, 55, 60])

print("mean  ", staff.mean())
print("median", staff.median())

The **mean** is the sum divided by the count. The **median** is the middle
value once the values are sorted: four salaries are below 47 and four above.
Here the two nearly agree, and either is a fair answer to "what does someone
earn at this company?".

Now add the owner, who pays herself 400.

In [ ]:
everyone = pd.Series([38, 41, 42, 45, 47, 50, 52, 55, 60, 400])

print("mean  ", everyone.mean())
print("median", everyone.median())
print("people who earn less than the mean:", (everyone < everyone.mean()).sum(), "of", len(everyone))

One value moved the mean from 47.8 to 83. Nine of the ten people earn less
than "the average salary". The median moved from 47 to 48.5, because with ten
values the middle falls between the fifth and the sixth.

Neither number is wrong. They answer different questions:

| The question | The number | Why |
|---|---|---|
| What does a typical person here earn? | median | half earn less, half earn more, whatever the extremes do |
| What does the company pay in total? | mean | mean × count is the total; the median knows nothing about totals |

Data with a long tail on one side is called **skewed**. Incomes, house prices,
city sizes, review counts and the areas of plant infestations are all skewed
to the right: most values are small and a few are enormous. Under skew, **the
mean is pulled toward the tail and the median stays put**. When the two
differ a lot, that itself is a finding: it tells you the data is skewed before
you have drawn anything.

Both by hand, so that nothing about them is magic:

In [ ]:
values = sorted(everyone)
count = len(values)

mean = sum(values) / count

middle = count // 2
if count % 2 == 1:
    median = values[middle]
else:
    median = (values[middle - 1] + values[middle]) / 2

print(mean, median)

**✏️ Try it:** Change the owner's 400 to 4,000. Which of the two numbers changes, and by how much? Then change it to 61.

---

## Part 2 — Spread: how far from the centre

Two cafés. In both, a customer waits five minutes on average.

In [ ]:
steady = pd.Series([4, 5, 5, 6, 5, 4, 6, 5])
erratic = pd.Series([1, 9, 2, 8, 5, 1, 10, 4])

print("steady   mean", steady.mean(), " median", steady.median())
print("erratic  mean", erratic.mean(), " median", erratic.median())

Same centre, and you would not mistake one café for the other. What differs is
the **spread**. The simplest measure is the range, largest minus smallest. It
depends on two values only, so one odd day decides it.

The **variance** uses every value. Take each value's distance from the mean,
square it so that distances below and above do not cancel, add the squares up,
and divide:

In [ ]:
distance = erratic - erratic.mean()
print(pd.DataFrame({"wait": erratic, "distance": distance,
                    "squared": distance ** 2}).to_string(index=False))
print()
total = (distance ** 2).sum()
print("sum of squares      ", total)
print("variance            ", total / (len(erratic) - 1))
print("standard deviation  ", (total / (len(erratic) - 1)) ** 0.5)

The variance is in *squared* minutes, which nobody can picture. Its square
root, the **standard deviation**, is back in minutes: at the erratic café a
wait is typically about 3.6 minutes away from the mean; at the steady one,
0.76.

**Why divide by count − 1 and not by count?** Dividing by count gives the
variance of exactly these eight numbers (the *population* variance). Usually
the eight are a sample of something larger, all the waits there ever were, and
then dividing by count comes out a little too small on average. Dividing by
count − 1 corrects that (the *sample* variance). With a thousand values the
two hardly differ. With eight they do.

What you must know is that **pandas and numpy disagree on the default**:

In [ ]:
print("erratic.var()              ", erratic.var())             # pandas: count - 1
print("np.var(erratic)            ", np.var(erratic))           # numpy:  count
print("np.var(erratic, ddof=1)    ", np.var(erratic, ddof=1))   # numpy, told to use count - 1

Same data, same function name, two answers, no warning. Whenever you report a
variance or a standard deviation, know which one you have.

### Quartiles and the interquartile range

The standard deviation has the mean's weakness: it is built from distances to
the mean, squared, so one extreme value inflates it. The median has a partner
that does not have that weakness.

Sort the values and cut them into four equal parts. The cutting points are the
**quartiles**: `q1` has a quarter of the values below it, `q2` is the median,
`q3` has three quarters below it. The distance from `q1` to `q3` is the
**interquartile range**: the width of the middle half of the data.

In [ ]:
for name, values in [("staff", staff), ("everyone", everyone)]:
    q1, q2, q3 = values.quantile([0.25, 0.5, 0.75])
    print("{:9} q1 {:6.2f}  median {:6.2f}  q3 {:6.2f}  iqr {:6.2f}  std {:7.2f}".format(
        name, q1, q2, q3, q3 - q1, values.std()))

Adding the owner multiplied the standard deviation by 15 and left the
interquartile range almost where it was.

| | Centre | Spread | One extreme value... |
|---|---|---|---|
| Built on sums | mean | variance, standard deviation | moves both |
| Built on order | median | quartiles, interquartile range | moves neither much |

`describe()` gives most of these at once. From now on you know what each of
its lines means.

In [ ]:
everyone.describe()

**✏️ Try it:** `steady.quantile(0.25)` is 4.75, a number that is not in the data. Sort the eight values and work out where 4.75 comes from.

---

## Part 3 — Outliers are questions

Is 400 "too far" from the other salaries? A common rule of thumb draws two
**fences**, one and a half interquartile ranges outside the quartiles, and
flags whatever lies beyond them.

In [ ]:
q1, q3 = everyone.quantile(0.25), everyone.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print("fences:", lower_fence, "and", upper_fence)

outside = (everyone < lower_fence) | (everyone > upper_fence)
pd.DataFrame({"salary": everyone, "outside the fences": outside})

The result is a column of True and False, as long as the data. **Nothing was
removed.** The rule marks a value for a person to look at. It cannot say what
the value is, and there are at least three things it can be:

| What it is | Example | What to do |
|---|---|---|
| A mistake | a salary typed as 400 where 40 was meant | correct it, or make it missing, and say so |
| A different kind of thing | the owner is not an employee | decide whether it belongs to the question |
| The most important row in the table | the one customer who brings half the revenue | keep it, and report it |

Deleting outliers because they are outliers gets rid of the third kind along
with the first. In Homework 8 the rule flags 160,543 of 863,064 plant
infestations, and those hold 97 % of the infested land.

1.5 is a convention, not a law. And on skewed data the rule flags a great
deal, because it quietly assumes that the two sides of the data look alike.

**✏️ Try it:** Flag the outliers of `erratic`. Are there any? Then of `pd.Series([3] * 9 + [4])` -- why is the 4 flagged?

---

## Part 4 — Your first figure

Ten numbers can be read. Two hundred cannot. Here are the waiting times of
200 customers at a third café (made-up data; how such numbers are generated is
Session 10).

In [ ]:
rng = np.random.default_rng(9)
waits = pd.Series(np.round(rng.gamma(shape=2.0, scale=2.5, size=200), 1))

print(waits.head(12).tolist())
print("mean", round(waits.mean(), 2), " median", waits.median(), " std", round(waits.std(), 2))

A **histogram** cuts the range of the values into intervals, called bins, and
draws one bar per bin, as tall as the number of values in it.

In [ ]:
fig, ax = plt.subplots()
ax.hist(waits, bins=20)
fig.tight_layout()

That is a picture, and it is not yet a result. What is on the x axis, in which
unit? What do the heights count? Before fixing it, look at the two names in
the first line.

- A **Figure** (`fig`) is the whole picture: the thing that is shown, and the
  thing that is saved to a file.
- An **Axes** (`ax`) is one plot inside it: a rectangle with an x axis and a y
  axis. A Figure can hold several.

`plt.subplots()` makes one of each. Everything after that is done by calling
methods of the Axes. This is matplotlib's **object interface**:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(waits, bins=20, color="steelblue", edgecolor="white")
ax.axvline(waits.mean(), color="firebrick", linestyle="--", label="mean {:.1f}".format(waits.mean()))
ax.axvline(waits.median(), color="black", linestyle=":", label="median {:.1f}".format(waits.median()))

ax.set_xlabel("waiting time (minutes)")
ax.set_ylabel("customers")
ax.set_title("A typical wait is about four minutes; a few customers wait three times as long")
ax.legend()
fig.tight_layout()

Now it can be read by someone who was not there when it was made. The mean
lies to the right of the median, pulled by the tail: Part 1, in a picture.

**The other way to plot, and why this course does not use it.** You will see
code everywhere that never mentions `ax`:

```python
plt.hist(waits)
plt.xlabel("waiting time (minutes)")
```

This is the *pyplot* interface. It draws on "the current Axes", a hidden
global that changes whenever a figure is created. It is fine for one quick
plot. It breaks down as soon as there are two plots, or the plotting is inside
a function, because the function cannot know which Axes is current. With the
object interface the Axes is a value you hold, pass to a function, and get
back.

| pyplot (avoid) | object interface (use) |
|---|---|
| `plt.hist(x)` | `ax.hist(x)` |
| `plt.xlabel("...")` | `ax.set_xlabel("...")` |
| `plt.title("...")` | `ax.set_title("...")` |
| `plt.xlim(0, 10)` | `ax.set_xlim(0, 10)` |

**A function that draws** therefore takes the Axes as an argument and returns
it:

In [ ]:
def draw_waits(ax, values, title):
    """Histogram of waiting times with the mean and the median marked."""
    ax.hist(values, bins=20, color="steelblue", edgecolor="white")
    ax.axvline(values.mean(), color="firebrick", linestyle="--", label="mean")
    ax.axvline(values.median(), color="black", linestyle=":", label="median")
    ax.set_xlabel("waiting time (minutes)")
    ax.set_ylabel("customers")
    ax.set_title(title)
    ax.legend()
    return ax


fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)
draw_waits(left, waits[:100], "The first hundred customers")
draw_waits(right, waits[100:], "The second hundred")
fig.tight_layout()

One Figure, two Axes, one function called twice. `sharex` and `sharey` give
both the same range, so the two can be compared by eye.

### Saving, and why nothing calls `show()`

In a notebook a figure appears when the cell that made it finishes. In a
**program**, run in a Terminal on the cluster, there is no screen and nothing
appears. A program saves its figures and closes them:

In [ ]:
import os
import tempfile

folder = tempfile.mkdtemp()
path = os.path.join(folder, "waits.png")

fig, ax = plt.subplots()
draw_waits(ax, waits, "Waiting times")
fig.savefig(path, dpi=120, bbox_inches="tight")
plt.close(fig)                      # free the memory; the notebook now shows nothing

print(path, os.path.getsize(path), "bytes")

`plt.close(fig)` matters in a program that draws many figures: matplotlib
keeps every one in memory until it is closed.

An Axes also **remembers** what was drawn on it, and you can ask it. This is
how a program can check a figure without anyone looking at it, and it is how
the tests of Homework 8 work.

In [ ]:
print("x label:", repr(ax.get_xlabel()))
print("title:  ", repr(ax.get_title()))
print("bars:   ", len(ax.patches))
print("x scale:", ax.get_xscale(), "  y axis from", ax.get_ylim()[0])

**✏️ Try it:** Draw the histogram of `waits` with 5 bins, then with 80. What does each one hide or invent?

---

## Part 5 — The chart follows the question

A coffee cart keeps a record of 84 days: the date, the temperature at noon,
and the number of cups sold (made-up data again).

In [ ]:
rng = np.random.default_rng(84)
days = pd.date_range("2026-06-01", periods=84, freq="D")
temperature = np.round(18 + 8 * np.sin(np.arange(84) / 13) + rng.normal(0, 3, 84), 1)
weekend = days.dayofweek >= 5
cups = np.round(260 - 5.5 * temperature + 60 * weekend + np.arange(84) * 0.9
                + rng.normal(0, 18, 84)).astype(int)

cart = pd.DataFrame({"date": days, "weekday": days.day_name(),
                     "temperature": temperature, "cups": cups})
cart.head()

Four different questions can be asked of this table, and each has a chart
that suits it.

| The question | Its kind | The chart | The summary number beside it |
|---|---|---|---|
| How many cups on a usual day, and how much does it vary? | **distribution** | histogram | median, quartiles |
| Which weekday sells most? | **comparison** | bars | one mean or median per group |
| Do cold days sell more? | **relationship** | scatter plot | correlation |
| Is the cart selling more than in June? | **trend** | line | change from first to last |

Decide which kind of question you have **before** choosing a chart. Most bad
charts are a good answer to a question nobody asked.

### Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.hist(cart["cups"], bins=15, color="steelblue", edgecolor="white")
ax.axvline(cart["cups"].median(), color="black", linestyle=":", label="median")
ax.set_xlabel("cups sold in a day")
ax.set_ylabel("days")
ax.set_title("Half of all days sell between 160 and 250 cups")
ax.legend()
fig.tight_layout()

### Comparison

One number per group, computed first, then one bar each. The order of the bars
is a decision: weekdays have a natural order, so use it. Groups with no natural
order are best sorted by height.

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = cart.groupby("weekday")["cups"].mean().reindex(order)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(by_day.index, by_day.values, color="steelblue")
ax.set_ylim(bottom=0)
ax.set_xlabel("day of the week")
ax.set_ylabel("mean cups sold")
ax.set_title("Weekends sell about a third more")
ax.tick_params(axis="x", labelrotation=30)
fig.tight_layout()

### Relationship

One point per day. A scatter plot is the only one of the four that shows every
row of the table.

In [ ]:
r = cart["temperature"].corr(cart["cups"])

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.scatter(cart["temperature"], cart["cups"], color="steelblue", alpha=0.7,
           label="r = {:.2f}".format(r))
ax.set_xlabel("temperature at noon (°C)")
ax.set_ylabel("cups sold")
ax.set_title("Colder days sell more coffee")
ax.legend()
fig.tight_layout()

The number in the legend is the **correlation**, r. It measures how closely
the points follow a straight line: 1 for a perfect rising line, −1 for a
perfect falling one, 0 when a line says nothing. From its definition:

In [ ]:
dx = cart["temperature"] - cart["temperature"].mean()
dy = cart["cups"] - cart["cups"].mean()

r_by_hand = (dx * dy).sum() / ((dx ** 2).sum() * (dy ** 2).sum()) ** 0.5
print(r_by_hand, r)

A day that is colder than usual (`dx` negative) and sells more than usual
(`dy` positive) contributes a negative product. Most days here do, so r is
negative.

Two cautions that the picture makes for you and the number does not. r
measures *straight-line* association only. And it says nothing about cause:
the cart also sells more on weekends, and if weekends happened to be colder,
temperature would get credit for what the weekend did.

### Trend

Time on the x axis, points joined in time order. A line says "this is one
thing changing", which is why it suits time and does not suit weekdays or
species.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(cart["date"], cart["cups"], color="lightsteelblue", label="each day")
ax.plot(cart["date"], cart["cups"].rolling(7, center=True).mean(), color="steelblue",
        linewidth=2.5, label="7-day mean")
ax.set_xlabel("date")
ax.set_ylabel("cups sold")
ax.set_title("Sales rose through the summer")
ax.legend()
fig.autofmt_xdate()

The pale line zigzags every week, because of the weekends. The 7-day mean
averages each day with its neighbours and shows the slower movement under the
zigzag. (`rolling` is in the reference notebook.)

### Same numbers, different pictures

Now the reason the picture is not optional. In 1973 the statistician Francis
Anscombe built four small datasets:

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quartet = pd.DataFrame({
    "dataset": ["I"] * 11 + ["II"] * 11 + ["III"] * 11 + ["IV"] * 11,
    "x": x123 * 3 + [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68,
          9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74,
          7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73,
          6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})

summary = quartet.groupby("dataset").agg(
    mean_x=("x", "mean"), mean_y=("y", "mean"), std_x=("x", "std"), std_y=("y", "std"))
summary["r"] = [rows["x"].corr(rows["y"]) for _, rows in quartet.groupby("dataset")]
summary

Five summary numbers, and the four datasets agree on all of them to two
decimals. By their numbers they are one dataset. Guess what they look like,
then run the next cell.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3), sharex=True, sharey=True)
for ax, (name, rows) in zip(axes, quartet.groupby("dataset")):
    ax.scatter(rows["x"], rows["y"], color="steelblue")
    ax.set_title("dataset " + name)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.tight_layout()

A loose cloud around a line. A curve. A perfect line with one point off it. A
vertical stack with one point that creates the whole correlation by itself.

The mean, the standard deviation and r cannot tell these apart, because all
three are built from sums, and many different shapes give the same sums.
**Always draw the data.** This is the picture Homework 8 has you make with
your own functions.

**✏️ Try it:** Compute the **median** of `y` for each of the four datasets. Do the medians agree as well as the means did?

---

## Part 6 — Honest pictures

A chart turns numbers into lengths, positions and slopes, and the reader's eye
turns them back. The chart is honest when the eye gets the right numbers.

### A bar's length is the number

A second cart, with steadier business, sold these cups in three months. The
same numbers, drawn twice:

In [ ]:
months = ["June", "July", "August"]
sold = [5210, 5420, 5580]

fig, (honest, cut) = plt.subplots(1, 2, figsize=(10, 3.6))
for ax in (honest, cut):
    ax.bar(months, sold, color="steelblue")
    ax.set_xlabel("month")
    ax.set_ylabel("cups sold")
cut.set_ylim(5100, 5600)
honest.set_title("Axis from zero: August is 7 % above June")
cut.set_title("Axis cut at 5,100: August looks 4 times June")
fig.tight_layout()

Nothing on the right is false. Every tick label is correct. But a bar says
"this much", and the right-hand bars say something the numbers do not. **Bars
start at zero.**

A line is different. A line shows *change*, by its slope, and its distance
from the bottom of the frame is not read as a quantity. A temperature chart
does not have to start at absolute zero. What a line can do instead is
exaggerate by its shape:

In [ ]:
fig, (wide, tall) = plt.subplots(1, 2, figsize=(10, 3.6),
                                 gridspec_kw={"width_ratios": [4, 1]})
for ax in (wide, tall):
    ax.plot(range(3), sold, marker="o", color="steelblue")
    ax.set_xticks(range(3))
    ax.set_xticklabels(["Jun", "Jul", "Aug"])
    ax.set_ylabel("cups sold")
wide.set_title("Wide frame: a steady rise")
tall.set_title("Narrow frame:\nsoaring")
fig.tight_layout()

Same three points. No rule fixes the right shape for a frame, so the honest
move is to put the size of the change in words, in the title or the caption:
"up 7 % from June to August".

### A logarithmic axis must fit the data, and be announced

Skewed data on an ordinary axis piles up in one corner. A **logarithmic** axis
gives every multiplication by ten the same distance. City populations, in
thousands:

In [ ]:
population = pd.Series([3, 5, 8, 9, 12, 15, 22, 30, 41, 55, 80, 120, 190, 400,
                        650, 1500, 3900, 8400])

fig, (plain, logged) = plt.subplots(1, 2, figsize=(10, 3.4))
plain.hist(population, bins=12, color="steelblue", edgecolor="white")
plain.set_title("Equal bins: one bar and some dust")

edges = np.geomspace(population.min(), population.max(), 12 + 1)
logged.hist(population, bins=edges, color="steelblue", edgecolor="white")
logged.set_xscale("log")
logged.set_title("Logarithmic axis: the shape appears")

for ax in (plain, logged):
    ax.set_ylabel("cities")
plain.set_xlabel("population (thousands)")
logged.set_xlabel("population (thousands, logarithmic scale)")
fig.tight_layout()

Two things to get right. The bin edges must be spaced evenly *on the
logarithmic scale* (`np.geomspace`), or the bars come out in wildly different
widths. And the label should say that the scale is logarithmic: a reader who
misses it will read distances as differences when they are ratios. Zero and
negative values have no place on such an axis and must be dealt with first.

### Labels and units

> A figure without axis labels and units is not yet a result.

A checklist to run on every figure before anyone else sees it:

| Check | Why |
|---|---|
| Both axes labelled, **with units** | "area" could be acres, hectares or square miles |
| A title that says what to see | "Weekends sell a third more", not "Cups by weekday" |
| Bars start at zero | a bar's length is the number |
| A logarithmic axis is announced | distances are ratios |
| A legend when there is more than one thing | and none when there is only one |
| Panels to be compared share their axes | otherwise each is stretched to fit itself |
| Nothing drawn that carries no data | 3-D effects, shadows, a rainbow of colours for one variable |

The first four can be checked by a program, and in Homework 8 you write that
program. The rest need a person.

**✏️ Try it:** Take the truncated bar chart above and make it honest in a different way: keep the cut axis, but draw **points** (`ax.plot(months, sold, 'o')`) in place of bars. Why is that acceptable?

---

## Exercises

All four use the `cart` table from Part 5.

### 🧩 Exercise 1 — Centre and spread

For `cart["cups"]`, compute the mean, the median, the standard deviation and
the interquartile range. Then do the same for weekdays and weekends
separately. Is the cart's day-to-day variation mostly a weekday/weekend
difference, or is there a lot of spread within each?

### 🧩 Exercise 2 — Fences

Flag the days whose `cups` lie outside the fences. How many are there? Does
that settle whether any day deserves a second look? List the five busiest
days with their date, weekday and temperature, and say what they have in
common.

### 🧩 Exercise 3 — A function that draws

Write `draw_scatter(ax, x, y, xlabel, ylabel)` that draws a scatter plot,
writes r in the legend with two decimals, labels both axes and returns `ax`.
Use it twice in one Figure: temperature against cups for weekdays on the left,
for weekends on the right, with shared axes. Is r stronger or weaker within
each group than for all days together? Why might that be?

### 🧩 Exercise 4 — Repair a figure

This code draws a figure that breaks four items of the checklist in Part 6.
Find them and fix them.

```python
fig, ax = plt.subplots()
by_day = cart.groupby("weekday")["cups"].mean()
ax.bar(by_day.index, by_day.values)
ax.set_ylim(150, 260)
```

---

## Summary

| Idea | The habit | The tool |
|---|---|---|
| Centre | Compute mean and median; if they differ, say why | `mean`, `median` |
| Spread | Know which variance you have | `var`, `std` (count − 1 in pandas, count in numpy), `quantile` |
| Outliers | Flag, look, then decide; never delete on sight | fences from `q1`, `q3` and 1.5 × iqr |
| The object interface | Hold the Axes; pass it to functions | `fig, ax = plt.subplots()`, `ax.set_...` |
| The chart follows the question | Name the kind of question first | `ax.hist`, `ax.bar`, `ax.scatter`, `ax.plot` |
| Honest pictures | Run the checklist | labels with units, bars from zero, shared axes |

### Where each idea appears in Homework 8

| Idea | Homework 8 |
|---|---|
| Mean, median, sample variance | Phase 1, `summary` |
| Quartiles, fences, flagging without deleting | Phase 2, `quartiles`, `flag_outliers` |
| Correlation from its definition; one summary per group | Phase 3, `correlation`, `summarize_groups` |
| A histogram with mean and median marked; a logarithmic axis | Phase 4, `plot_distribution` |
| Bars from zero, a line in time order, a scatter plot with r | Phase 5, `plot_comparison`, `plot_trend`, `plot_relationship` |
| Several Axes with shared ranges; the checklist as a program | Phase 6, `small_multiples`, `audit_figure` |
| Saving and closing | Phase 7, `build_figures`, `save_figures` |

**Next:** `02_Consumer_Complaints_Example` describes 1.3 million real records
this way, and `03_Statistics_and_Plotting_Reference` is the lookup guide.